In [ ]:
!unzip -q "/content/drive/MyDrive/DS 4002 - Project 3/MILK10k_Training_Input.zip" -d /content/MILK10k_images

# **11-Classifications, image + metadata**

In [ ]:
import os, random, time, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# ── 1. CONFIGURATION & PATHS ──────────────────────────────────
SAVE_DIR_MM = Path('/content/drive/MyDrive/Model_results') # Changed to user requested directory
SAVE_DIR_MM.mkdir(parents=True, exist_ok=True)

IMAGE_DIR       = Path('/content/MILK10k_images/MILK10k_Training_Input')
METADATA_CSV    = Path('/content/drive/MyDrive/DS 4002 - Project 3/MILK10k_Training_Metadata.csv')
GROUNDTRUTH_CSV = Path('/content/drive/MyDrive/DS 4002 - Project 3/MILK10k_Training_GroundTruth.csv')

CFG = dict(
    img_size      = 224,
    batch_size    = 32,
    num_workers   = 2,
    num_epochs    = 15, # Changed to 15 epochs
    lr            = 1e-4,
    weight_decay  = 1e-4,
    dropout       = 0.4,
    test_split    = 0.3, # Changed to 0.3 for 70/15/15 split (validation and test each get 15%)
    seed          = 42,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(CFG['seed'])

# ── 2. DATA PREPARATION (11-CLASS MULTIMODAL) ─────────────────
meta = pd.read_csv(METADATA_CSV)
gt   = pd.read_csv(GROUNDTRUTH_CSV)

ORIG_CLASS_COLS = ['AKIEC', 'BCC', 'BEN_OTH', 'BKL', 'DF', 'INF', 'MAL_OTH', 'MEL', 'NV', 'SCCKA', 'VASC']

gt['class_name'] = gt[ORIG_CLASS_COLS].idxmax(axis=1)
gt['label'] = gt['class_name'].apply(lambda x: ORIG_CLASS_COLS.index(x))

df = meta.merge(gt[['lesion_id', 'label']], on='lesion_id', how='inner')
df['age_approx'] = df['age_approx'].fillna(df['age_approx'].mean())
df['site'] = df['site'].fillna('unknown')

df_encoded = pd.get_dummies(df, columns=['site', 'sex'], dummy_na=False)
meta_cols = [c for c in df_encoded.columns if c.startswith(('site_', 'sex_', 'age_approx'))]
META_DIM = len(meta_cols)

# 3-WAY STRATIFIED SPLIT (70/15/15)
unique_lesions = df_encoded.drop_duplicates(subset='lesion_id')
train_l, temp_l = train_test_split(unique_lesions['lesion_id'], test_size=CFG['test_split'],
                                   stratify=unique_lesions['label'], random_state=CFG['seed'])
val_l, test_l = train_test_split(temp_l, test_size=0.50,
                                 stratify=unique_lesions[unique_lesions['lesion_id'].isin(temp_l)]['label'],
                                 random_state=CFG['seed'])

train_df = df_encoded[df_encoded['lesion_id'].isin(train_l)].reset_index(drop=True)
val_df   = df_encoded[df_encoded['lesion_id'].isin(val_l)].reset_index(drop=True)
test_df  = df_encoded[df_encoded['lesion_id'].isin(test_l)].reset_index(drop=True)

# ── 3. DATASET & TRANSFORMATIONS ─────────────────────────────
class ISICDatasetMM(Dataset):
    def __init__(self, dataframe, image_dir, meta_cols, transform=None):
        self.df         = dataframe
        self.image_dir  = Path(image_dir)
        self.meta_cols  = meta_cols
        self.transform  = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = None
        for ext in ('.jpg', '.jpeg', '.png', '.JPG'):
            p = self.image_dir / str(row['lesion_id']) / f"{row['isic_id']}{ext}"
            if p.exists():
                img = Image.open(p).convert('RGB')
                break
        if img is None:
            img = Image.new('RGB', (CFG['img_size'], CFG['img_size']))

        if self.transform:
            img = self.transform(img)

        meta_data = torch.tensor(row[self.meta_cols].values.astype(np.float32))
        return img, meta_data, int(row['label'])

train_tfm = transforms.Compose([
    transforms.Resize((CFG['img_size']+32, CFG['img_size']+32)),
    transforms.RandomCrop(CFG['img_size']),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    # Removed transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_tfm = transforms.Compose([
    transforms.Resize((CFG['img_size'], CFG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ── 4. MODEL (LATE FUSION - 11 CLASSES) ──────────────────────
class ISICModelMM11(nn.Module):
    def __init__(self, num_classes=11, meta_dim=META_DIM, dropout=0.4):
        super().__init__()
        # num_classes=0 returns pooled features directly — consistent with image-only model
        self.backbone = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        img_dim = self.backbone.num_features

        self.meta_net = nn.Sequential(
            nn.Linear(meta_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout / 2)
        )
        self.head = nn.Sequential(
            nn.Linear(img_dim + 64, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x, m):
        x = self.backbone(x)   # uses timm's built-in global pooling
        m = self.meta_net(m)
        return self.head(torch.cat((x, m), dim=1))

# ── 5. TRAINING HELPERS ──────────────────────────────────────
def run_epoch_mm_11(model, loader, criterion, optimizer=None, phase='train'):
    model.train() if phase == 'train' else model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_labels, all_probs = [], []

    with torch.set_grad_enabled(phase == 'train'):
        for imgs, metas, labels in loader:
            imgs, metas, labels = imgs.to(DEVICE), metas.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs, metas)
            loss = criterion(logits, labels)

            if phase == 'train':
                optimizer.zero_grad(); loss.backward(); optimizer.step()

            probs = torch.softmax(logits, dim=1)
            running_loss += loss.item() * imgs.size(0)
            correct += (probs.argmax(dim=1) == labels).sum().item()
            total += imgs.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

    all_labels_np = np.array(all_labels)
    all_probs_np  = np.array(all_probs)

    present_classes = np.unique(all_labels_np)
    valid_auc_labels = []
    for c in present_classes:
        if len(np.unique((all_labels_np == c).astype(int))) > 1:
            valid_auc_labels.append(c)

    if len(valid_auc_labels) > 1:
        auc = roc_auc_score(
            all_labels_np,
            all_probs_np,
            multi_class='ovr',
            labels=valid_auc_labels
        )
    else:
        auc = np.nan

    return running_loss / total, correct / total, auc, all_labels, all_probs

# ── 6. EXECUTION ──────────────────────────────────────────────
train_loader = DataLoader(ISICDatasetMM(train_df, IMAGE_DIR, meta_cols, train_tfm),
                          batch_size=CFG['batch_size'],
                          sampler=WeightedRandomSampler(1./np.bincount(train_df['label'])[train_df['label'].values], len(train_df)),
                          num_workers=CFG['num_workers'])
val_loader  = DataLoader(ISICDatasetMM(val_df,  IMAGE_DIR, meta_cols, eval_tfm), batch_size=CFG['batch_size'], shuffle=False)
test_loader = DataLoader(ISICDatasetMM(test_df, IMAGE_DIR, meta_cols, eval_tfm), batch_size=CFG['batch_size'], shuffle=False)

model = ISICModelMM11(num_classes=11).to(DEVICE)

# Freeze backbone initially for first 3 epochs
for param in model.backbone.parameters():
    param.requires_grad = False
print("Backbone frozen for initial head training.")

# Optimizer for only trainable parts (meta_net and head)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['num_epochs'])
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

history  = {'epoch':[], 'tr_auc':[], 'vl_auc':[], 'tr_loss':[], 'vl_loss':[], 'tr_acc':[], 'vl_acc':[]}
best_auc = 0.0

for epoch in range(1, CFG['num_epochs'] + 1):
    t0 = time.time()

    # Unfreeze backbone after 3 epochs and re-initialize optimizer
    if epoch == 4: # After epoch 3, unfreeze
        print("\nUnfreezing backbone and re-initializing optimizer for full model fine-tuning.")
        for param in model.backbone.parameters():
            param.requires_grad = True
        # Re-initialize optimizer to include backbone parameters with differential LRs
        optimizer = optim.AdamW([
            {'params': model.backbone.parameters(), 'lr': 5.5e-5},
            {'params': model.meta_net.parameters(), 'lr': CFG['lr']},
            {'params': model.head.parameters(), 'lr': CFG['lr']}
        ], weight_decay=CFG['weight_decay'])
        # The scheduler will continue its cycle based on the total number of epochs

    tr_loss, tr_acc, tr_auc, _, _ = run_epoch_mm_11(model, train_loader, criterion, optimizer, 'train')
    vl_loss, vl_acc, vl_auc, _, _ = run_epoch_mm_11(model, val_loader,   criterion, None,      'val')
    scheduler.step()

    for k, v in zip(history.keys(), [epoch, tr_auc, vl_auc, tr_loss, vl_loss, tr_acc, vl_acc]):
        history[k].append(v)

    print(f"--- Epoch {epoch:02d} Summary ---")
    print(f"TRAIN | Loss: {tr_loss:.4f} | Acc: {tr_acc:.4f} | AUC: {tr_auc:.4f}")
    print(f"VAL   | Loss: {vl_loss:.4f} | Acc: {vl_acc:.4f} | AUC: {vl_auc:.4f}")
    print(f"Time  | {time.time()-t0:.1f}s")

    if vl_auc > best_auc:
        best_auc = vl_auc
        print(f"✅ Saving best model based on Val AUC: {best_auc:.4f}\n")
        torch.save(model.state_dict(), SAVE_DIR_MM / 'best_mm_model_11class.pth')
    else:
        print("\n")

# ── 7. FINAL TEST EVALUATION ──────────────────────────────────
print("\n--- Final Evaluation on Unseen Test Set (11-Class Multimodal) ---")
model.load_state_dict(torch.load(SAVE_DIR_MM / 'best_mm_model_11class.pth', weights_only=True))
ts_loss, ts_acc, ts_auc, ts_labels, ts_probs = run_epoch_mm_11(model, test_loader, criterion, phase='test')

ts_preds = np.array(ts_probs).argmax(axis=1)

# Recalculate valid_auc_labels for the test set for verification
present_test_classes = np.unique(ts_labels)
valid_test_auc_labels = []
for c in present_test_classes:
    if len(np.unique((np.array(ts_labels) == c).astype(int))) > 1:
        valid_test_auc_labels.append(c)

print(f"Test AUC (One-vs-Rest): {ts_auc:.4f}")
print(f"Test Acc: {ts_acc:.4f}")
print(classification_report(ts_labels, ts_preds, target_names=ORIG_CLASS_COLS, labels=list(range(len(ORIG_CLASS_COLS)))))

# Export Results
pd.DataFrame(history).to_csv(SAVE_DIR_MM / 'mm_history_11class.csv', index=False)

cm = confusion_matrix(ts_labels, ts_preds, labels=list(range(len(ORIG_CLASS_COLS))))
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=ORIG_CLASS_COLS, yticklabels=ORIG_CLASS_COLS)
plt.title('Test Confusion Matrix (11-Class Multimodal)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.savefig(SAVE_DIR_MM / 'test_cm_11class_mm.png')
plt.show()

epochs = history['epoch']
plt.figure(figsize=(15,5))
plt.subplot(1,3,1); plt.plot(epochs, history['tr_auc'], label='Train'); plt.plot(epochs, history['vl_auc'], label='Val'); plt.title('AUC (OVR)'); plt.xlabel('Epoch'); plt.ylabel('AUC'); plt.legend()
plt.subplot(1,3,2); plt.plot(epochs, history['tr_loss'], label='Train'); plt.plot(epochs, history['vl_loss'], label='Val'); plt.title('Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()
plt.subplot(1,3,3); plt.plot(epochs, history['tr_acc'], label='Train'); plt.plot(epochs, history['vl_acc'], label='Val'); plt.title('Accuracy'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend()
plt.tight_layout(); plt.savefig(SAVE_DIR_MM / 'curves_11class_mm.png'); plt.show()

Backbone frozen for initial head training.
--- Epoch 01 Summary ---
TRAIN | Loss: 1.9154 | Acc: 0.3969 | AUC: 0.8160
VAL   | Loss: 1.8960 | Acc: 0.3963 | AUC: 0.7743
Time  | 61.1s
✅ Saving best model based on Val AUC: 0.7743

--- Epoch 02 Summary ---
TRAIN | Loss: 1.5659 | Acc: 0.5669 | AUC: 0.9010
VAL   | Loss: 1.8288 | Acc: 0.4141 | AUC: 0.7845
Time  | 61.0s
✅ Saving best model based on Val AUC: 0.7845

--- Epoch 03 Summary ---
TRAIN | Loss: 1.4421 | Acc: 0.6231 | AUC: 0.9213
VAL   | Loss: 1.8250 | Acc: 0.4090 | AUC: 0.7927
Time  | 59.2s
✅ Saving best model based on Val AUC: 0.7927


Unfreezing backbone and re-initializing optimizer for full model fine-tuning.
--- Epoch 04 Summary ---
TRAIN | Loss: 1.2497 | Acc: 0.7031 | AUC: 0.9472
VAL   | Loss: 1.6654 | Acc: 0.4796 | AUC: 0.8230
Time  | 64.5s
✅ Saving best model based on Val AUC: 0.8230

--- Epoch 05 Summary ---
TRAIN | Loss: 1.0903 | Acc: 0.7672 | AUC: 0.9644
VAL   | Loss: 1.5847 | Acc: 0.5178 | AUC: 0.8338
Time  | 64.3s
✅ Saving 